# 03_gap_analysis.ipynb

ユーザーの会話から「満たされていないニーズ (Gap)」や「潜在的な不満」を抽出します。
特定のアーキタイプ（例：不満を持っているユーザー）に絞って分析することで、製品改善のヒントを得ることができます。

In [ ]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src.loader.nucc_loader import NUCCLoader
from src.preprocessor.tokenizer import Tokenizer
from src.analysis.archetype import ArchetypeEngine
from src.analysis.gap_analysis import GapAnalysisEngine

In [ ]:
# 1. データの準備 (NUCC)
data_path = Path.cwd().parent / 'data/raw/nucc/nucc'
loader = NUCCLoader(data_dir=str(data_path))
df = loader.load()

if not df.empty:
    print("Tokenizing...")
    tokenizer = Tokenizer()
    df['tokenized_text'] = df['text'].apply(lambda x: tokenizer.tokenize(x))
    
    # アーキタイプ分類を実行してラベルを付与
    arch_engine = ArchetypeEngine(n_clusters=4)
    features = arch_engine.analyze_user_characteristics(df)
    labeled_df = arch_engine.classify_archetypes(features)
    
    # 元のデータフレームにラベルを結合
    df = df.merge(labeled_df[['archetype_label']], on='user_id', how='left')
    
    print("Data prepared with Archetypes:")
    display(df['archetype_label'].value_counts())

## 2. ギャップ分析の実行
会話の中から「欲しい」「困る」「使いにくい」といったシグナルを検出します。

In [ ]:
if not df.empty:
    gap_engine = GapAnalysisEngine()
    
    # 全体のギャップ分析
    gaps_df = gap_engine.analyze_gaps(df)
    
    print(f"Found {len(gaps_df)} potential gaps/complaints.")
    
    if not gaps_df.empty:
        # 上位の不満ワード
        all_keywords = []
        for kws in gaps_df['matched_keywords']:
            all_keywords.extend(kws)
            
        kw_counts = pd.Series(all_keywords).value_counts().head(10)
        
        plt.figure(figsize=(10, 6))
        sns.barplot(x=kw_counts.values, y=kw_counts.index, palette='rocket')
        plt.title('Top Gap Signals (Keywords)')
        plt.xlabel('Count')
        plt.show()
        
        # 具体的な発言例を表示
        print("\n--- Context Examples ---")
        for i, row in gaps_df.sample(min(5, len(gaps_df))).iterrows():
            print(f"[{row['insight_type']}] User: {row['user_id']}\nText: {row['text']}\nKeywords: {row['matched_keywords']}\n")

## 3. 特定アーキタイプへのフォーカス
例えば「批判的(Critical)」なユーザーの発言に絞って分析します。

In [ ]:
target_label = 'Vocal Critic (Advisor)' # ArchetypeEngineの出力ラベル名に合わせる
# ラベル名が存在するか確認
labels = df['archetype_label'].unique()
print(f"Available labels: {labels}")

# 存在しそうなラベルを適当に選ぶ（デモ用）
if not df.empty:
    target = labels[0] if len(labels) > 0 else None
    
    if target:
        print(f"Analyzing gaps for: {target}")
        target_gaps = gap_engine.analyze_gaps(df, target_archetype=target)
        
        if not target_gaps.empty:
            display(target_gaps[['user_id', 'text', 'matched_keywords']].head())
        else:
            print("No gaps found for this group.")